# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moriyamao/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane, Refresh / Content Opportunity Scoring, is a **scoring / ranking** task, with a **classification** model underneath it. The goal is not just "yes/no is this page declining" in isolation — it's to produce a ranked list of pages, ordered by how urgently they deserve review. A classifier's predicted probability (e.g. probability of decline) becomes the score used to rank. This matches the lane guide directly: the output is "a ranked review queue with scores, actions, and reason codes," and precision@K is the natural way to judge a ranking, not plain accuracy.

In [1]:
print("Task type: scoring / ranking, built on top of a binary classification model (decline probability -> rank score)")

Task type: scoring / ranking, built on top of a binary classification model (decline probability -> rank score)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The provisional target is `is_declining_label = (trend_direction == "down")`. This label comes from an **observed** measurement (real traffic/impressions data bucketed into a trend direction), not from a hand-defined business rule like `health_score` or `priority_score` (those are explicitly excluded from this dataset). That said, per the lane guide, this is still a **proxy label**: `trend_direction` is calculated from the current window, not a true future outcome. A stronger version of this target, planned for later weeks, would be: features from the prior 90 days -> decline observed over the next 30 days. I'm using the simpler current-window proxy now because it's what the starter dataset ships, and I'm naming the limitation explicitly rather than hiding it.

In [2]:
print("Target: is_declining_label = (trend_direction == \'down\') -- an observed proxy, not yet a future-window label")

Target: is_declining_label = (trend_direction == 'down') -- an observed proxy, not yet a future-window label


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The metric is **Precision@50**: of the top 50 pages the model ranks highest for review, what fraction are actually declining? This matches how the output would really be used -- a reviewer with limited time works through a capped list, not the whole inventory, so top-K precision reflects the real decision better than overall accuracy or ROC-AUC alone. From the starter pipeline's own verified results, the baseline hand-written rule scores 0.240 at Precision@50, while a random forest scores 0.740 -- meaning the learned ranking gets roughly 37 of the top 50 right versus about 12 for the rule. That gap is the number I'd need my own model to reproduce or beat to call this lane's approach worthwhile.

In [3]:
print("Success metric: Precision@50 -- baseline rule = 0.240, random forest (starter pipeline) = 0.740")

Success metric: Precision@50 -- baseline rule = 0.240, random forest (starter pipeline) = 0.740


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row in this dataset = **one content page** (one `content_id`, deduplicated), with its own visibility, staleness, and trend signals for the current window. That's the grain the whole lane operates at: I am ranking individual pages, not clients or days. The cell below loads the starter slice, confirms the row count, and shows what the target column actually looks like next to a few of the raw signals it's derived from.

In [4]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df.drop_duplicates(subset="content_id")

print("Rows (one row = one content page):", len(df))
print("Columns:", df.shape[1])

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

df[["content_id", "trend_direction", "is_declining_label", "impressions_90d",
    "days_since_last_update", "content_age_days"]].head(8)

Rows (one row = one content page): 30000
Columns: 44


,content_id,trend_direction,is_declining_label,impressions_90d,days_since_last_update,content_age_days
0,content_304f48230142,down,1,3803,20,187
1,content_a1fb4e703a9e,down,1,15320,25,445
2,content_9aa793d4d895,down,1,12581,20,141
3,content_331d6c4de07b,stable,0,11751,22,463
4,content_d99b7a2d90ca,down,1,19140,14,263
5,content_d4084a4bc775,down,1,3970,20,147
6,content_9a34b442b552,down,1,20,20,90
7,content_a63219c6e95a,stable,0,1724,22,445


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple rule like "flag pages that are old and haven't been updated in 180+ days" sounds reasonable, but the data itself pushes back on it: in this starter slice, declining pages are actually **younger on average (~236 days) than stable or growing pages (~290 days)**. A single-threshold age rule would misclassify a large share of real cases, because decline here isn't driven by one clean variable -- it's a mix of staleness, visibility, position, and content depth interacting together, and those interactions shift depending on content type and traffic level. That's exactly the situation the lane guide describes as "a plain rule isn't enough" -- the pattern is real (the starter pipeline's random forest clearly beats the hand-written baseline, 0.740 vs 0.240 Precision@50), but it's too tangled to hand-write as a short if-statement without either missing real decline cases or flooding the queue with false positives.

In [5]:
print("Avg content_age_days by trend_direction:")
print(df.groupby("trend_direction")["content_age_days"].mean().round(0))
print()
print("A single age threshold would misclassify many pages -- decline is not simply \'old content\'")

Avg content_age_days by trend_direction:
trend_direction
down      236.0
flat      246.0
new       239.0
stable    295.0
up        288.0
Name: content_age_days, dtype: float64

A single age threshold would misclassify many pages -- decline is not simply 'old content'


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.